# Tutorial · A model is an object: `.fit`, `.predict`, `.score`

**Today:** by the end of this tutorial you can read a line from the slides such as `cross_val_score(BernoulliNB(), X[:, :5], y, cv=folds)` and say what each piece is: a model object made by a call, a splitter object, and a function that runs the unit 7 lab's fold loop on them. You can make a scikit-learn model, fit it to a table, read the counts it learned, score it on held-out rows, and write a loop over a splitter's folds that fits and scores any model.

**Before you start:** scikit-learn is not yet in the `bf550` environment. Install it once, from a terminal where your environment lives:

```
conda install -n bf550 -c conda-forge scikit-learn
```

The problem sets from PS8 on ship an `environment.yml` that lists it, so the install happens once. This notebook has its own repository, separate from the unit labs. Run `gh student accept bu-bioinfo-classrooms bf550-fall-2026 bf550-scikit-learn-tutorial`, clone what it creates, and open the notebook with the `bf550` kernel. The setup cell below builds the same ten isolates the unit 7 lab uses, so nothing from that notebook is needed here. The unit 7 lab's sections are cited by number where this notebook builds on them.

Each section names one idea, explains what it does, and asks you to **predict
what a cell prints before you run it**. Write the prediction down, on paper, out
loud, or in a comment. A prediction you can compare against the output is what
tells you which parts of the code you can already read.

Three sections end with a **Test your understanding** task: write a small piece
of code, then run the check cell under it. Every task has a hint in the **Hints**
block at the end of the notebook, for when you want it. The check never grades and never
breaks anything. A ⬜ means not attempted yet, a ❌ means not yet and comes with
a hint, and a ✅ means passing. Run the check cells rather than editing them.
Everything else in the notebook is yours to change.

**AI in this tutorial.** Until your prediction is written down, work at level 1, with
no AI. The prediction is how you find out what you can read unaided, and the
final exam is level 1. Once you have run a cell, level 3 is encouraged: ask your
tutor to explain anything you missed.

Run every cell, and change things to see what happens. Nothing in this notebook
can be broken in a way that matters.

## Setup

Run this cell first. It imports NumPy, the four scikit-learn names the tutorial uses, and the self-check, and it builds the unit 7 lab's two objects. `X` is a table of ten isolates and four genes, one row per isolate, with `True` where the isolate carries the gene. `y` holds the growth test's result for each isolate, `True` for resistant.

In [ ]:
import numpy as np
from sklearn.naive_bayes import BernoulliNB
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from labcheck import check

# ten isolates (rows) and four genes (columns); True = the isolate carries the gene
X = np.array([[True,  True, True,  True ],
              [False, True, False, False],
              [False, True, True,  True ],
              [False, True, False, True ],
              [True,  True, True,  True ],
              [True,  True, False, False],
              [False, True, True,  False],
              [False, True, False, False],
              [False, True, True,  True ],
              [False, True, False, False]])
# the growth test for each isolate; True = resistant, and the two classes take turns
y = np.array([True, False, True, False, True, False, True, False, True, False])

print("setup done")

## 1 · You have been using objects all along

Every value in Python is an **object**, a value that carries its own functions and its own stored values. A function that belongs to an object is a **method**, and it is called with a dot after the object's name. `rng.random()` draws from the generator `rng`, `scores.append(0.5)` adds to the list `scores`, and `X.mean(axis=0)` averages the table `X`. A stored value that belongs to an object is an **attribute**, read with the dot and no parentheses, as in `X.shape`. What a method does depends on what kind of object it belongs to, and `type(thing).__name__` prints that kind.

A method reads and changes the object it belongs to and no other. Two generators made with the same seed are two objects, and each keeps its own position in its own sequence.

**Predict before you run.** What are the three names on the first line? Does the third line repeat the second or continue it? What does the fourth line print?

In [ ]:
rng = np.random.default_rng(7)
print(type(rng).__name__, type(X).__name__, type([]).__name__)
print(rng.random(3).round(3))
other = np.random.default_rng(7)
print(other.random(3).round(3))
print(rng.random(3).round(3))

`Generator ndarray list`, then `[0.625 0.897 0.776]` twice, then `[0.225 0.3   0.874]`. The third line repeats the second because `other` is a separate object that started where `rng` started. The fourth line continues `rng`'s own sequence, which `other`'s draws did not advance. The unit 7 lab's `shuffled_copy` relied on this when it took `rng` as a parameter. The function received the object and drew from it, and the caller's generator moved on.

## 2 · A model is made by a call, with its settings in the parentheses

scikit-learn's classifiers are objects. `BernoulliNB()` makes one. The name of the kind of object is followed by parentheses, in the same way that `np.random.default_rng(7)` makes a generator. Settings go inside the parentheses as keyword arguments, `name=value`. `BernoulliNB` is the naive Bayes of units 5 to 7, built on a present/absent table, and the one setting that matters here is `alpha`, the pseudocount the slides mentioned. Its default is 1.

A freshly made model has learned nothing. It holds its settings and nothing else. Printing the object shows its kind and any setting that differs from the default, and the method `get_params()` returns every setting as a dictionary.

**Predict before you run.** What prints for `model`, and what for the model made with `alpha=0.5`? How many entries does the dictionary have?

In [ ]:
model = BernoulliNB()
print(model)
print(BernoulliNB(alpha=0.5))
print(model.get_params())
print(type(model).__name__)

`BernoulliNB()` prints as the call that made it, and the second line shows the one setting that was changed. The dictionary has five entries. `alpha` is 1.0, and the other four stay at their defaults throughout this course. The variable `model` now holds an object the way `rng` holds a generator, and nothing has been fitted.

**Modify.** Change `alpha=0.5` to `alpha=1` and predict the second line before you run it. The printed form shows only what differs from the default.

## 3 · `.fit` learns from a table and its labels, in place

`model.fit(X, y)` is the method that trains. It takes the table, one row per isolate and one column per gene, and the labels, one per row, and it stores what it learned inside the object. The stored values are attributes whose names end in an underscore. `model.classes_` lists the labels in the order the model uses, `model.class_count_` is how many training rows each class had, and `model.feature_count_` holds, for each class in that order, how many of its rows carried each gene. The unit 7 lab's section 1.1 computed each gene's share within a class; the counts divided by the class count are those shares.

`fit` also returns the object it just fitted. The slides use this when they write `BernoulliNB().fit(X, y)` in one line. The call makes the model, then fits it, and the result is the fitted model.

**Predict before you run.** What are `classes_` and `class_count_`? Which class is the second row of `feature_count_`, and what is that row divided by its class count? Section 1.1 printed `[0.4 1. 1. 0.8]` for the resistant isolates.

In [ ]:
model = BernoulliNB()
fitted = model.fit(X, y)
print(fitted is model)                                    # the same object came back
print(model.classes_, model.class_count_)
print(model.feature_count_)
print(model.feature_count_[1] / model.class_count_[1])     # shares within the second class

`True`: `fit` returned the very object it changed, so `fitted` and `model` are two names for one fitted model. The classes are `[False True]`, so in every per-class attribute the first row is the susceptible class and the second the resistant one. The second row of `feature_count_`, `[2 5 5 4]`, is how many of the five resistant isolates carry each gene, and divided by 5 it is section 1.1's `[0.4 1. 1. 0.8]`.

The table the model uses when it predicts is `feature_log_prob_`, the log of the smoothed share, with `alpha` added to the count and twice `alpha` to the class count. `np.exp(model.feature_log_prob_)` shows the smoothed shares, and the first entry is 2 / 7 = 0.286.

**Modify.** Fit the same `model` again on the first six isolates only, `model.fit(X[:6], y[:6])`, and predict `class_count_` before you print it. A second `fit` replaces what the first one learned, and nothing accumulates.

## 4 · `.predict` calls one row at a time, and `.score` counts the matches

`model.predict(rows)` returns one call per row of `rows`, from the table the model learned, with `True` for resistant. It takes a table even for a single isolate, so one row is written `X[[0]]` or `X[0:1]`, which keep the two dimensions. `X[0]` is a one-dimensional row and raises an error. `model.predict_proba(rows)` returns, for each row, one probability per class in `classes_` order, which is unit 5's posterior. `model.score(rows, labels)` fits nothing. It calls every row and returns the share of calls that match `labels`, which is unit 6's accuracy. The unit 7 lab's section 2.2 wrote that share as `(calls == test_y).mean()`.

**Predict before you run.** What are the calls for the first three isolates? Gene 2 separates the classes in this table. Which column of the probabilities is larger in each row? What is the score on all ten, and is the last line `True`?

In [ ]:
model = BernoulliNB().fit(X, y)
print(model.predict(X[:3]))
print(model.predict_proba(X[:3]).round(3))
print(model.score(X, y))
print(model.score(X, y) == (model.predict(X) == y).mean())

`[ True False  True]`: isolates 0 and 2 carry gene 2 and are called resistant, and isolate 1 lacks it and is called susceptible. Column 0 holds the probability of `False` and column 1 of `True`, in `classes_` order, and each row sums to 1. The score on all ten is 1.0. That number is a training score, because the model is scored on the rows it learned from, and the unit's first session showed how far such a score can sit from the score on new isolates. The last line is `True` because `score` is `predict` followed by the mean of the matches.

**Modify.** Change `X[:3]` in the second line to `X[0]`, read the error, then put it back. The message says what shape was expected.

## 5 · The same calls on a split

Training rows and held-out rows need no new method. Fit on one slice and score on the other. The unit 7 lab's section 1.2 shuffled and then sliced; here the first six isolates train and the last four are held out, as in its section 3.3. `fit` reads only the six rows it is given, so the four held-out labels never enter the model.

**Predict before you run.** What are the calls on the four held-out isolates, against `y[6:]`, and what is the score? Isolate 6 is resistant and lacks genes 0 and 3.

In [ ]:
model = BernoulliNB().fit(X[:6], y[:6])
print(model.predict(X[6:]))
print(y[6:])
print(model.score(X[6:], y[6:]))

Three of the four calls match, so the score is 0.75. Isolate 6 is called susceptible. It carries gene 2, which the six training isolates tie to resistance, but it lacks genes 0 and 3, which they also tie to resistance, and the two absences outweigh the one presence. One miss in four is a score of 0.75, and a split this small is why the unit moved on to folds.

**Test your understanding.** Write a function named `holdout_score` that takes four parameters: `model`, a scikit-learn model that has not been fitted yet; `X`, a table with one row per isolate; `y`, its labels, one per row; and `n_train`, how many rows from the top are the training rows. It fits `model` on the first `n_train` rows and returns one number, `model`'s score on the remaining rows. This is question 1. Its hint is at the end of the notebook.

In [ ]:
# your turn: a function named holdout_score(model, X, y, n_train)

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("holdout_score", expect=0.75, args=(BernoulliNB(), X, y, 6),
      hint="section 5's cell is the body, with 6 replaced by n_train and the score returned")
check("holdout_score", expect=1.0, args=(BernoulliNB(), X, y, 8),
      hint="rows 8 and 9 are the held-out rows: a resistant isolate with gene 2 and a susceptible one without it")

## 6 · A splitter is an object too, and `cross_val_score` runs the fold loop

The unit 7 lab's section 2.1 dealt fold ids in turn. scikit-learn keeps the folds in a **splitter** object, `StratifiedKFold(n_splits=5)`. Like a model before `fit`, it stores only its settings. Its method `folds.split(X, y)` produces the folds, one pair per fold. Each pair holds the positions of the training rows and the positions of the test rows. A `for` loop over the pairs reads both names at once, as a loop over `zip` does. *Stratified* means every fold gets the same mix of classes as the whole table. With the classes taking turns in `y` and no shuffle, each fold's test rows are two consecutive isolates. The `shuffle=True, random_state=7` the slides add shuffles the rows first, as section 1.2 did with random keys.

**Predict before you run.** Which two rows does each fold hold out?

In [ ]:
folds = StratifiedKFold(n_splits=5)
for train_rows, test_rows in folds.split(X, y):
    print("train", train_rows, "test", test_rows)

Each fold's test rows are two consecutive isolates, one of each class, and its training rows are the other eight. The pairs are positions, so `X[test_rows]` is the test table and `y[test_rows]` its labels, as section 2.2's masks `fold_id == k` picked them.

The loop below is section 2.2's loop with the splitter supplying the rows and the model supplying the score. `cross_val_score` is that loop as one call. It takes the model, the table, the labels and the splitter, fits a fresh copy of the model in every fold, and returns the test scores.

**Predict before you run.** Of the five scores, which fold holds isolate 6, and what does it score? Does `cross_val_score` return the same five numbers?

In [ ]:
scores = []
for train_rows, test_rows in folds.split(X, y):
    model = BernoulliNB().fit(X[train_rows], y[train_rows])
    scores.append(model.score(X[test_rows], y[test_rows]))
print(np.array(scores), np.mean(scores))
print(cross_val_score(BernoulliNB(), X, y, cv=folds))

`[1.  1.  1.  0.5 1. ]` with a mean of 0.9, and `cross_val_score` returns the same five numbers. Fold 3 holds isolates 6 and 7 and scores 0.5, because isolate 6 is called susceptible, as in section 5. The slides' line `cross_val_score(BernoulliNB(), X[:, :5], y, cv=folds)` reads the same way, with `X[:, :5]` keeping the first five columns of the slides' larger table.

**Test your understanding.** Write a function named `fold_scores_with` that takes four parameters: `model`, a scikit-learn model that has not been fitted yet; `X`, a table with one row per isolate; `y`, its labels; and `folds`, a splitter object such as `StratifiedKFold(n_splits=5)`. It returns a list with one number per fold, in the order `folds.split(X, y)` gives them: `model`'s score on that fold's test rows, after fitting it on the fold's training rows. The loop above is the body. This is question 2. Its hint is at the end of the notebook.

In [ ]:
# your turn: a function named fold_scores_with(model, X, y, folds)

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("fold_scores_with", expect=[1.0, 1.0, 1.0, 0.5, 1.0],
      args=(BernoulliNB(), X, y, StratifiedKFold(n_splits=5)),
      hint="section 6's loop computes exactly this list")
check("fold_scores_with", expect=[1.0, 0.5, 1.0, 0.5, 1.0],
      args=(BernoulliNB(), X[:, [3]], y, StratifiedKFold(n_splits=5)),
      hint="X[:, [3]] is a one-column table; gene 3 alone misses isolates 3 and 6, which sit in folds 1 and 3")

## 7 · Any object with the same methods fits the same slot

`cross_val_score` and your `fold_scores_with` call only `fit` and `score`, so they run on any model that has those methods, which is every classifier in scikit-learn. `DummyClassifier(strategy="most_frequent")` is unit 6's all-one-class baseline as an object. Its `fit` finds the most common training label, and its `predict` returns that label for every row. Next unit's `DecisionTreeClassifier` and unit 9's `LogisticRegression` are made, fitted and scored with the same three calls. What changes from one kind of model to the next is the settings in the parentheses and the names of the fitted attributes, and `help(BernoulliNB)` in a cell, or the model's documentation page, lists both.

**Predict before you run.** The first six isolates are three of each class, and a tie goes to the first entry of `classes_`. What does the baseline call the four held-out isolates, what does it score, and what are its five cross-validation scores?

In [ ]:
baseline = DummyClassifier(strategy="most_frequent").fit(X[:6], y[:6])
print(baseline.classes_, baseline.predict(X[6:]), baseline.score(X[6:], y[6:]))
print(cross_val_score(DummyClassifier(strategy="most_frequent"), X, y, cv=folds))

`[False  True]`, four calls of `False`, and a score of 0.5. The tie goes to `False`, so every held-out isolate is called susceptible, and half of the calls match. Every fold scores 0.5 for the same reason. The model object changed from naive Bayes to the baseline, and the three calls stayed the same.

**Test your understanding.** Create a variable named `gene3_cv` that stores one number, the mean of the five cross-validation scores from `cross_val_score` of a `BernoulliNB()` on gene 3 alone, the one-column table `X[:, [3]]`, with the labels `y` and the splitter `StratifiedKFold(n_splits=5)`. The unit 7 lab's section 2.2 found 0.8 for this gene with its own folds. This is question 3. Its hint is at the end of the notebook.

In [ ]:
# your turn: a variable named gene3_cv

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("gene3_cv", expect=0.8,
      hint="cross_val_score returns five numbers; the variable stores their mean")

### If you finish early

- Run `BernoulliNB().predict(X)` on a model that was never fitted, and read the error. It names the method that was skipped.
- Give `fold_scores_with` the slides' splitter, `StratifiedKFold(n_splits=5, shuffle=True, random_state=7)`, and say before you run it whether the five scores can still be read off section 6's fold table.
- Refit on `X` and `y` with `alpha=0.5`, and predict which entries of `np.exp(model.feature_log_prob_)` move toward section 1.1's plain shares before you print them.
- PS7 is open.

### If you are stuck

Wave someone over. This hour exists so that a stuck step costs you a minute
rather than an evening. Known snags:

- **`ModuleNotFoundError: No module named 'sklearn'`.** scikit-learn is not installed in the environment the kernel runs. Run the install from *Before you start* in a terminal, then restart the kernel.
- **`ValueError: Expected 2D array, got 1D array`.** `fit`, `predict` and `score` take a table. Write `X[[6]]` for one row and `X[:, [3]]` for one gene; both keep the second dimension.
- **`NotFittedError`.** `predict` or `score` ran on a model that was never fitted, or on a different object from the one that was. Fit the model you score, under the same name.
- **`holdout_score` or `fold_scores_with` returns `None`.** `fit` changes the model in place and hands back the model, never a number. Return the score from a `return` line.
- **`fold_scores_with` passes the first check and fails the second.** The second check's `X` has one column. Slice rows only, `X[train_rows]` and `X[test_rows]`, and leave the columns alone.

## Hints

**Question 1 · `holdout_score`.** Section 5's cell is the body. Call `model.fit(X[:n_train], y[:n_train])`, then return `model.score(X[n_train:], y[n_train:])`.

**Question 2 · `fold_scores_with`.** Section 6's loop is the body, with the function's own `model`, `X`, `y` and `folds`. Start `scores = []`, loop `for train_rows, test_rows in folds.split(X, y):`, fit `model` on `X[train_rows]` and `y[train_rows]`, `append` its score on `X[test_rows]` and `y[test_rows]`, and return `scores` after the loop, at the loop's own indentation.

**Question 3 · `gene3_cv`.** Section 6's last line is the call, with `X[:, [3]]` in place of `X` and `StratifiedKFold(n_splits=5)` as `cv`. The call returns five numbers; store their `.mean()`.